# Notebook 4 — Ablation Study

Prereq: Make sure notebook 2 has already produced the YOLO classification datasets. It can run locally or in Colab as long as the dataset folders are available. In Colab, it first looks for Notebook 2 outputs in `/content`; if they are not there but were saved to `/drive/MyDrive`, it restores them automatically.

Expected input datasets:

- `yolo_dataset_v2`
- `yolo_dataset_v2_spatial`
- `yolo_dataset_baseline`
- `yolo_dataset_baseline_spatial`

The notebook reuses existing ablation runs when present and trains missing runs when `ALLOW_TRAINING = True`.

In [ ]:
# Optional dependency install. Keep this cell for portability across local machines and Colab.
# It intentionally does not install torch; use a GPU-enabled torch build for your hardware.
%pip install -q ultralytics tqdm pandas numpy matplotlib seaborn scikit-learn pillow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import gc
import random
import shutil
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision.models as tv_models
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from ultralytics import YOLO
from sklearn.metrics import f1_score

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

PROJECT_ROOT = Path.cwd()
RUNS_DIR = PROJECT_ROOT / "runs" / "ablation_study"
RESULTS_DIR = PROJECT_ROOT / "ablation_results"
ABLATION_DATASETS_DIR = PROJECT_ROOT / "ablation_datasets"
WEIGHTS_DIR = PROJECT_ROOT / "yolo_weights"
for path in [RUNS_DIR, RESULTS_DIR, ABLATION_DATASETS_DIR, WEIGHTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# If runs already exist, they are reused. Missing runs are trained only when this is True.
ALLOW_TRAINING = True
FORCE_RETRAIN = False

EPOCHS = 10
BATCH = 16
PATIENCE = 5
NUM_WORKERS = 0

DEVICE = 0 if torch.cuda.is_available() else "cpu"
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
REQUIRED_SPLITS = ["train", "val", "test"]
REQUIRED_CLASSES = ["warehouse", "non_warehouse"]
POSITIVE_CLASS = "warehouse"

print(f"Project root: {PROJECT_ROOT}")
print(f"Runs dir:     {RUNS_DIR}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Device:       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"ALLOW_TRAINING={ALLOW_TRAINING}, FORCE_RETRAIN={FORCE_RETRAIN}, EPOCHS={EPOCHS}, BATCH={BATCH}")

c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3
Runs dir:     c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\runs\ablation_study
Results dir:  c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\ablation_results
Device:       NVIDIA GeForce RTX 5070 Ti
ALLOW_TRAINING=True, FORCE_RETRAIN=False, EPOCHS=10, BATCH=16


In [ ]:
DATASET_FOLDER_NAMES = [
    "yolo_dataset_v2",
    "yolo_dataset_v2_spatial",
    "yolo_dataset_baseline",
    "yolo_dataset_baseline_spatial",
]


def has_dataset_set(base: Path) -> bool:
    return all((base / name / "manifest.csv").exists() for name in DATASET_FOLDER_NAMES)


def restore_datasets_from_drive(target_base: Path) -> Path | None:
    """Restore Notebook 2 datasets saved by its Drive backup cell, when available."""
    drive_base = Path("/drive/MyDrive")
    if not drive_base.exists():
        try:
            from google.colab import drive
            drive.mount("/drive")
        except Exception:
            return None
    if not drive_base.exists():
        return None
    if not all((drive_base / name).exists() for name in DATASET_FOLDER_NAMES):
        return None

    target_base.mkdir(parents=True, exist_ok=True)
    for name in DATASET_FOLDER_NAMES:
        src = drive_base / name
        dst = target_base / name
        if dst.exists():
            continue
        print(f"Restoring {name}: {src} -> {dst}")
        shutil.copytree(src, dst)
    return target_base if has_dataset_set(target_base) else None


def find_datasets_dir(project_root: Path) -> Path:
    """Find Notebook 2 dataset outputs in Colab or local layouts."""
    candidates = [
        Path("/content"),
        project_root / "content",
        project_root / "datasets",
        project_root,
    ]
    for base in candidates:
        if has_dataset_set(base):
            return base

    restore_target = Path("/content") if Path("/content").exists() else project_root / "content"
    restored = restore_datasets_from_drive(restore_target)
    if restored is not None:
        return restored

    raise FileNotFoundError(
        "Could not find Notebook 2 datasets. Expected yolo_dataset_* folders under "
        "/content, content/, datasets/, project root, or /drive/MyDrive."
    )


DATASETS_DIR = find_datasets_dir(PROJECT_ROOT)
DATASET_ROOTS = {
    "baseline_random": DATASETS_DIR / "yolo_dataset_baseline",
    "density_random": DATASETS_DIR / "yolo_dataset_v2",
    "density_spatial": DATASETS_DIR / "yolo_dataset_v2_spatial",
    "baseline_spatial": DATASETS_DIR / "yolo_dataset_baseline_spatial",
}


def count_images(root: Path) -> pd.DataFrame:
    rows = []
    for split in REQUIRED_SPLITS:
        for cls in REQUIRED_CLASSES:
            cls_dir = root / split / cls
            rows.append({
                "dataset": root.name,
                "split": split,
                "class": cls,
                "n_images": len(list(cls_dir.glob("*.png"))) if cls_dir.exists() else 0,
            })
    return pd.DataFrame(rows)

counts = []
for key, root in DATASET_ROOTS.items():
    if not root.exists():
        raise FileNotFoundError(f"Missing {root}. Run Notebook 2 through image download, augmentation, and manifest saving first.")
    manifest = root / "manifest.csv"
    if not manifest.exists():
        raise FileNotFoundError(f"Missing manifest: {manifest}")
    df = count_images(root).assign(dataset_key=key)
    if df["n_images"].sum() == 0:
        raise RuntimeError(f"Dataset has no images: {root}")
    counts.append(df)

counts_df = pd.concat(counts, ignore_index=True)
display(counts_df.pivot_table(index=["dataset_key", "split"], columns="class", values="n_images", aggfunc="sum"))
print(f"Using Notebook 2 datasets from: {DATASETS_DIR}")

class                   non_warehouse  warehouse
dataset_key      split                          
baseline_random  test             305        842
                 train           3668       2522
                 val              306        841
baseline_spatial test             298        788
                 train           3728       2629
                 val              298        788
density_random   test             305        842
                 train           3668       2522
                 val              306        841
density_spatial  test             298        788
                 train           3728       2629
                 val              298        788

Using Notebook 2 datasets from: c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\content


In [ ]:
def prepare_no_aug_dataset(src_root: Path, dst_root: Path) -> Path:
    """Create a copy of the density dataset with offline augmentation files removed."""
    if dst_root.exists() and not FORCE_RETRAIN:
        print(f"Using existing no-augmentation dataset: {dst_root}")
        return dst_root
    if dst_root.exists():
        shutil.rmtree(dst_root)

    for split in REQUIRED_SPLITS:
        for cls in REQUIRED_CLASSES:
            src_dir = src_root / split / cls
            dst_dir = dst_root / split / cls
            dst_dir.mkdir(parents=True, exist_ok=True)
            copied = 0
            for src in src_dir.glob("*.png"):
                if "_aug" in src.stem:
                    continue
                shutil.copy2(src, dst_dir / src.name)
                copied += 1
            print(f"{dst_root.name}/{split}/{cls}: copied {copied}")

    for metadata_name in ["manifest.csv", "dataset.yaml"]:
        src = src_root / metadata_name
        if src.exists():
            shutil.copy2(src, dst_root / metadata_name)
    return dst_root

DENSITY_NO_AUG_ROOT = prepare_no_aug_dataset(
    DATASET_ROOTS["density_random"],
    ABLATION_DATASETS_DIR / "yolo_dataset_v2_no_aug",
)

YOLO_EXPERIMENTS = [
    {"name": "01_yolov8n_640_density", "question": "YOLOv8n capacity baseline", "family": "YOLO", "dataset_label": "density-aware negatives", "dataset": DATASET_ROOTS["density_random"], "model": "yolov8n-cls.pt", "imgsz": 640, "dropout": 0.5, "weight_decay": 1e-3, "label_smoothing": 0.1, "freeze": None},
    {"name": "02_yolov8s_320_density", "question": "YOLOv8s at 320px", "family": "YOLO", "dataset_label": "density-aware negatives", "dataset": DATASET_ROOTS["density_random"], "model": "yolov8s-cls.pt", "imgsz": 320, "dropout": 0.5, "weight_decay": 1e-3, "label_smoothing": 0.1, "freeze": None},
    {"name": "03_yolov8s_640_no_aug", "question": "YOLOv8s without offline augmentation", "family": "YOLO", "dataset_label": "density-aware negatives, no aug", "dataset": DENSITY_NO_AUG_ROOT, "model": "yolov8s-cls.pt", "imgsz": 640, "dropout": 0.5, "weight_decay": 1e-3, "label_smoothing": 0.1, "freeze": None},
    {"name": "04_yolov8s_640_weak_reg", "question": "YOLOv8s weak regularization", "family": "YOLO", "dataset_label": "density-aware negatives", "dataset": DATASET_ROOTS["density_random"], "model": "yolov8s-cls.pt", "imgsz": 640, "dropout": 0.0, "weight_decay": 1e-4, "label_smoothing": 0.0, "freeze": None},
    {"name": "05_yolov8s_640_strong_reg", "question": "YOLOv8s stronger regularization", "family": "YOLO", "dataset_label": "density-aware negatives", "dataset": DATASET_ROOTS["density_random"], "model": "yolov8s-cls.pt", "imgsz": 640, "dropout": 0.5, "weight_decay": 1e-3, "label_smoothing": 0.0, "freeze": None},
    {"name": "06_yolov8s_640_strong_reg_ls", "question": "YOLOv8s strong regularization + label smoothing", "family": "YOLO", "dataset_label": "density-aware negatives", "dataset": DATASET_ROOTS["density_random"], "model": "yolov8s-cls.pt", "imgsz": 640, "dropout": 0.5, "weight_decay": 1e-3, "label_smoothing": 0.1, "freeze": None},
    {"name": "08_yolov8s_640_baseline_negatives", "question": "YOLOv8s baseline-global negatives", "family": "YOLO", "dataset_label": "baseline global negatives", "dataset": DATASET_ROOTS["baseline_random"], "model": "yolov8s-cls.pt", "imgsz": 640, "dropout": 0.5, "weight_decay": 1e-3, "label_smoothing": 0.1, "freeze": None},
]
RESNET_EXPERIMENT = {"name": "09_resnet18_224_density", "question": "ResNet18 architecture baseline", "family": "ResNet18", "dataset_label": "density-aware negatives", "dataset": DATASET_ROOTS["density_random"], "imgsz": 224}

display(pd.DataFrame([{k: str(v) if isinstance(v, Path) else v for k, v in exp.items() if k != "dataset"} | {"dataset": str(exp["dataset"])} for exp in YOLO_EXPERIMENTS + [RESNET_EXPERIMENT]]))

yolo_dataset_v2_no_aug/train/warehouse: copied 2522
yolo_dataset_v2_no_aug/train/non_warehouse: copied 917
yolo_dataset_v2_no_aug/val/warehouse: copied 841
yolo_dataset_v2_no_aug/val/non_warehouse: copied 306
yolo_dataset_v2_no_aug/test/warehouse: copied 842
yolo_dataset_v2_no_aug/test/non_warehouse: copied 305


,name,question,family,dataset_label,model,imgsz,dropout,weight_decay,label_smoothing,freeze,dataset
0,01_yolov8n_640_density,YOLOv8n capacity baseline,YOLO,density-aware negatives,yolov8n-cls.pt,640,0.5,0.0010,0.1,NaN,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
1,02_yolov8s_320_density,YOLOv8s at 320px,YOLO,density-aware negatives,yolov8s-cls.pt,320,0.5,0.0010,0.1,NaN,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
2,03_yolov8s_640_no_aug,YOLOv8s without offline augmentation,YOLO,"density-aware negatives, no aug",yolov8s-cls.pt,640,0.5,0.0010,0.1,NaN,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
3,04_yolov8s_640_weak_reg,YOLOv8s weak regularization,YOLO,density-aware negatives,yolov8s-cls.pt,640,0.0,0.0001,0.0,NaN,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
4,05_yolov8s_640_strong_reg,YOLOv8s stronger regularization,YOLO,density-aware negatives,yolov8s-cls.pt,640,0.5,0.0010,0.0,NaN,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
5,06_yolov8s_640_strong_reg_ls,YOLOv8s strong regularization + label smoothing,YOLO,density-aware negatives,yolov8s-cls.pt,640,0.5,0.0010,0.1,NaN,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
6,08_yolov8s_640_baseline_negatives,YOLOv8s baseline-global negatives,YOLO,baseline global negatives,yolov8s-cls.pt,640,0.5,0.0010,0.1,NaN,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
7,09_resnet18_224_density,ResNet18 architecture baseline,ResNet18,density-aware negatives,NaN,224,NaN,NaN,NaN,NaN,c:\Users\zz\Desktop\school\SCM 256\final_proj\...


In [ ]:
def clear_torch_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def train_or_reuse_yolo(exp: dict):
    run_dir = RUNS_DIR / exp["name"]
    weights = run_dir / "weights" / "best.pt"
    results_csv = run_dir / "results.csv"
    if weights.exists() and results_csv.exists() and not FORCE_RETRAIN:
        print(f"[reuse] {exp['name']} -> {weights}")
        return weights
    if not ALLOW_TRAINING:
        print(f"[missing; skipped training] {exp['name']} -> {weights}")
        return None

    print(f"\n[train] {exp['name']} | {exp['question']}")
    clear_torch_memory()
    model = YOLO(exp["model"])
    kwargs = dict(
        data=str(exp["dataset"]),
        epochs=EPOCHS,
        imgsz=exp["imgsz"],
        batch=BATCH,
        patience=PATIENCE,
        dropout=exp["dropout"],
        weight_decay=exp["weight_decay"],
        label_smoothing=exp["label_smoothing"],
        project=str(RUNS_DIR),
        name=exp["name"],
        exist_ok=True,
        seed=RANDOM_STATE,
        device=DEVICE,
        workers=NUM_WORKERS,
        verbose=False,
    )
    if exp.get("freeze") is not None:
        kwargs["freeze"] = exp["freeze"]
    model.train(**kwargs)
    clear_torch_memory()
    if not weights.exists():
        raise FileNotFoundError(f"Expected YOLO weights not found: {weights}")
    if not results_csv.exists():
        raise FileNotFoundError(f"Expected YOLO results not found: {results_csv}")
    return weights

for exp in YOLO_EXPERIMENTS:
    exp["weights"] = train_or_reuse_yolo(exp)

print("YOLO ablation runs are ready.")


[train] 01_yolov8n_640_density | YOLOv8n capacity baseline
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.48  Python-3.13.7 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\content\yolo_dataset_v2, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.5, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01

c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\.venv\Lib\site-packages\torch\nn\modules\module.py:1370: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


AMP: checks passed 
train: Fast image access  (ping: 0.10.0 ms, read: 740.8225.9 MB/s, size: 579.8 KB)
train: Scanning C:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\content\yolo_dataset_v2\train... 6190 images, 0 corrupt: 100% ━━━━━━━━━━━━ 6190/6190 1.7Kit/s 3.5s<0.0s
train: New cache created: C:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\content\yolo_dataset_v2\train.cache
val: Fast image access  (ping: 0.10.0 ms, read: 553.682.3 MB/s, size: 301.8 KB)
val: Scanning C:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\content\yolo_dataset_v2\val... 1147 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1147/1147 1.5Kit/s 0.8s0.1s
val: New cache created: C:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\content\yolo_dataset_v2\val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automat

In [ ]:
def make_resnet_loaders(root: Path, imgsz: int = 224):
    train_tf = T.Compose([
        T.Resize((imgsz, imgsz)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_tf = T.Compose([
        T.Resize((imgsz, imgsz)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    train_ds = ImageFolder(root / "train", transform=train_tf)
    val_ds = ImageFolder(root / "val", transform=eval_tf)
    test_ds = ImageFolder(root / "test", transform=eval_tf)
    loaders = {
        "train": DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS),
        "val": DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS),
        "test": DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS),
    }
    return loaders, train_ds.classes


def evaluate_resnet_epoch(model, loader, criterion, positive_idx):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(TORCH_DEVICE), y.to(TORCH_DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    acc = float(np.mean(np.array(y_true) == np.array(y_pred)))
    f1 = f1_score(y_true, y_pred, pos_label=positive_idx, zero_division=0)
    return total_loss / len(loader.dataset), acc, f1


def train_or_reuse_resnet(exp: dict):
    run_dir = RUNS_DIR / exp["name"]
    run_dir.mkdir(parents=True, exist_ok=True)
    weights = run_dir / "best.pt"
    history_path = WEIGHTS_DIR / "resnet18_density_v2_history.csv"
    run_history_path = run_dir / "history.csv"

    if weights.exists() and history_path.exists() and not FORCE_RETRAIN:
        print(f"[reuse] {exp['name']} -> {weights}")
        return weights
    if not ALLOW_TRAINING:
        print(f"[missing; skipped training] {exp['name']} -> {weights}")
        return None

    print(f"\n[train] {exp['name']} | {exp['question']}")
    clear_torch_memory()
    loaders, classes = make_resnet_loaders(exp["dataset"], exp["imgsz"])
    positive_idx = classes.index(POSITIVE_CLASS)

    model = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, len(classes)))
    model.to(TORCH_DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

    best_val_acc = -1.0
    patience_left = PATIENCE
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        for x, y in tqdm(loaders["train"], desc=f"ResNet epoch {epoch}/{EPOCHS}", leave=False):
            x, y = x.to(TORCH_DEVICE), y.to(TORCH_DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)
            train_correct += (logits.argmax(dim=1) == y).sum().item()

        train_loss /= len(loaders["train"].dataset)
        train_acc = train_correct / len(loaders["train"].dataset)
        val_loss, val_acc, val_f1 = evaluate_resnet_epoch(model, loaders["val"], criterion, positive_idx)
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc, "val_f1": val_f1})
        print(f"epoch={epoch:02d} train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_left = PATIENCE
            torch.save({"model_state": model.state_dict(), "classes": classes, "imgsz": exp["imgsz"]}, weights)
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stopping at epoch {epoch}")
                break

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)
    history_df.to_csv(run_history_path, index=False)
    clear_torch_memory()
    return weights

RESNET_EXPERIMENT["weights"] = train_or_reuse_resnet(RESNET_EXPERIMENT)
print("ResNet ablation baseline is ready.")


[train] 09_resnet18_224_density | ResNet18 architecture baseline


epoch=01 train_loss=0.4234 train_acc=0.8648 val_loss=0.4045 val_acc=0.8657 val_f1=0.9096


epoch=02 train_loss=0.3045 train_acc=0.9504 val_loss=0.4046 val_acc=0.8814 val_f1=0.9187


epoch=03 train_loss=0.2672 train_acc=0.9753 val_loss=0.3805 val_acc=0.8989 val_f1=0.9322


epoch=04 train_loss=0.2455 train_acc=0.9861 val_loss=0.3938 val_acc=0.8823 val_f1=0.9209


epoch=05 train_loss=0.2359 train_acc=0.9924 val_loss=0.3745 val_acc=0.8980 val_f1=0.9316


epoch=06 train_loss=0.2276 train_acc=0.9968 val_loss=0.4155 val_acc=0.8762 val_f1=0.9184


epoch=07 train_loss=0.2250 train_acc=0.9964 val_loss=0.3975 val_acc=0.8814 val_f1=0.9173


epoch=08 train_loss=0.2249 train_acc=0.9964 val_loss=0.3777 val_acc=0.8989 val_f1=0.9316
Early stopping at epoch 8
ResNet ablation baseline is ready.


In [ ]:
def summarize_yolo_run(exp: dict):
    results_path = RUNS_DIR / exp["name"] / "results.csv"
    if not results_path.exists():
        print(f"[missing results] {exp['name']} -> {results_path}")
        return None
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()
    if "metrics/accuracy_top1" not in df.columns:
        raise KeyError(f"Unexpected YOLO results columns in {results_path}: {list(df.columns)}")
    best_idx = df["metrics/accuracy_top1"].idxmax()
    best = df.loc[best_idx]
    return {
        "experiment": exp["name"],
        "question": exp["question"],
        "model_family": exp["family"],
        "dataset": exp["dataset_label"],
        "best_epoch": int(best["epoch"]),
        "best_val_acc": float(best["metrics/accuracy_top1"]),
        "best_val_loss": float(best["val/loss"]),
        "final_val_acc": float(df.iloc[-1]["metrics/accuracy_top1"]),
        "final_val_loss": float(df.iloc[-1]["val/loss"]),
        "epochs_completed": int(df.iloc[-1]["epoch"]),
        "source": str(results_path),
    }


def summarize_resnet_run(exp: dict):
    history_path = WEIGHTS_DIR / "resnet18_density_v2_history.csv"
    if not history_path.exists():
        print(f"[missing ResNet history] {history_path}")
        return None
    df = pd.read_csv(history_path)
    best_idx = df["val_acc"].idxmax()
    best = df.loc[best_idx]
    return {
        "experiment": exp["name"],
        "question": exp["question"],
        "model_family": exp["family"],
        "dataset": exp["dataset_label"],
        "best_epoch": int(best["epoch"]),
        "best_val_acc": float(best["val_acc"]),
        "best_val_loss": float(best["val_loss"]),
        "final_val_acc": float(df.iloc[-1]["val_acc"]),
        "final_val_loss": float(df.iloc[-1]["val_loss"]),
        "epochs_completed": int(df.iloc[-1]["epoch"]),
        "source": str(history_path),
    }

rows = []
for exp in YOLO_EXPERIMENTS:
    row = summarize_yolo_run(exp)
    if row is not None:
        rows.append(row)
resnet_row = summarize_resnet_run(RESNET_EXPERIMENT)
if resnet_row is not None:
    rows.append(resnet_row)

if not rows:
    raise RuntimeError("No ablation metrics found. Enable ALLOW_TRAINING or provide runs/ablation_study logs.")

metrics_df = pd.DataFrame(rows).sort_values("best_val_acc", ascending=False).reset_index(drop=True)
metrics_path = RESULTS_DIR / "report_ready_ablation_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
# Compatibility with Notebook 3.5 naming.
metrics_df.to_csv(RESULTS_DIR / "ablation_metrics.csv", index=False)
print(f"Saved: {metrics_path}")
display(metrics_df)

Saved: c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\ablation_results\report_ready_ablation_metrics.csv


,experiment,question,model_family,dataset,best_epoch,best_val_acc,best_val_loss,final_val_acc,final_val_loss,epochs_completed,source
0,08_yolov8s_640_baseline_negatives,YOLOv8s baseline-global negatives,YOLO,baseline global negatives,9,0.919790,0.248960,0.916300,0.284990,10,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
1,04_yolov8s_640_weak_reg,YOLOv8s weak regularization,YOLO,density-aware negatives,8,0.911940,0.281280,0.911070,0.323540,10,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
2,02_yolov8s_320_density,YOLOv8s at 320px,YOLO,density-aware negatives,10,0.906710,0.293250,0.906710,0.293250,10,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
3,03_yolov8s_640_no_aug,YOLOv8s without offline augmentation,YOLO,"density-aware negatives, no aug",10,0.903230,0.249990,0.903230,0.249990,10,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
4,09_resnet18_224_density,ResNet18 architecture baseline,ResNet18,density-aware negatives,3,0.898867,0.380459,0.898867,0.377693,8,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
5,01_yolov8n_640_density,YOLOv8n capacity baseline,YOLO,density-aware negatives,8,0.891020,0.321160,0.865740,0.357960,10,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
6,06_yolov8s_640_strong_reg_ls,YOLOv8s strong regularization + label smoothing,YOLO,density-aware negatives,1,0.890150,0.294350,0.887530,0.320310,6,c:\Users\zz\Desktop\school\SCM 256\final_proj\...
7,05_yolov8s_640_strong_reg,YOLOv8s stronger regularization,YOLO,density-aware negatives,1,0.890150,0.294350,0.887530,0.320310,6,c:\Users\zz\Desktop\school\SCM 256\final_proj\...


In [ ]:
def get_metric(experiment: str, metric: str = "best_val_acc"):
    match = metrics_df.loc[metrics_df["experiment"] == experiment, metric]
    if match.empty:
        return None
    return float(match.iloc[0])

REPORT_COMPARISONS = [
    {
        "claim": "Best YOLO configuration outperforms ResNet18 baseline",
        "before": "09_resnet18_224_density",
        "after": metrics_df[metrics_df["model_family"] == "YOLO"].iloc[0]["experiment"],
    },
    {
        "claim": "Density-aware negative sampling outperforms baseline-global negatives in the matched YOLOv8s setting",
        "before": "08_yolov8s_640_baseline_negatives",
        "after": "06_yolov8s_640_strong_reg_ls",
    },
    {
        "claim": "YOLOv8s at 320px compared with YOLOv8n at 640px",
        "before": "01_yolov8n_640_density",
        "after": "02_yolov8s_320_density",
    },
    {
        "claim": "Offline augmentation effect in YOLOv8s 640px setting",
        "before": "03_yolov8s_640_no_aug",
        "after": "06_yolov8s_640_strong_reg_ls",
    },
    {
        "claim": "Strong regularization compared with weak regularization",
        "before": "04_yolov8s_640_weak_reg",
        "after": "05_yolov8s_640_strong_reg",
    },
    {
        "claim": "Label smoothing added to strong regularization",
        "before": "05_yolov8s_640_strong_reg",
        "after": "06_yolov8s_640_strong_reg_ls",
    },
]

comparison_rows = []
for item in REPORT_COMPARISONS:
    before_acc = get_metric(item["before"])
    after_acc = get_metric(item["after"])
    if before_acc is None or after_acc is None:
        print(f"[skip comparison] {item['claim']} because one side is missing")
        continue
    comparison_rows.append({
        "claim": item["claim"],
        "before": item["before"],
        "after": item["after"],
        "before_best_val_acc": before_acc,
        "after_best_val_acc": after_acc,
        "delta_best_val_acc": after_acc - before_acc,
        "supported": after_acc > before_acc,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = RESULTS_DIR / "report_ready_ablation_claims.csv"
comparison_df.to_csv(comparison_path, index=False)
# Compatibility with Notebook 3.5 naming.
comparison_df.to_csv(RESULTS_DIR / "ablation_pairwise_deltas.csv", index=False)
print(f"Saved: {comparison_path}")
display(comparison_df)

Saved: c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\ablation_results\report_ready_ablation_claims.csv


,claim,before,after,before_best_val_acc,after_best_val_acc,delta_best_val_acc,supported
0,Best YOLO configuration outperforms ResNet18 b...,09_resnet18_224_density,08_yolov8s_640_baseline_negatives,0.898867,0.91979,0.020923,True
1,Density-aware negative sampling outperforms ba...,08_yolov8s_640_baseline_negatives,06_yolov8s_640_strong_reg_ls,0.919790,0.89015,-0.029640,False
2,YOLOv8s at 320px compared with YOLOv8n at 640px,01_yolov8n_640_density,02_yolov8s_320_density,0.891020,0.90671,0.015690,True
3,Offline augmentation effect in YOLOv8s 640px s...,03_yolov8s_640_no_aug,06_yolov8s_640_strong_reg_ls,0.903230,0.89015,-0.013080,False
4,Strong regularization compared with weak regul...,04_yolov8s_640_weak_reg,05_yolov8s_640_strong_reg,0.911940,0.89015,-0.021790,False
5,Label smoothing added to strong regularization,05_yolov8s_640_strong_reg,06_yolov8s_640_strong_reg_ls,0.890150,0.89015,0.000000,False


In [ ]:
plot_df = metrics_df.sort_values("best_val_acc", ascending=True)
plt.figure(figsize=(10, 5))
plt.barh(plot_df["experiment"], plot_df["best_val_acc"])
plt.xlabel("Best validation accuracy")
plt.title("Report-ready ablation results from completed training logs")
plt.xlim(0, 1)
plt.tight_layout()
plot_path = RESULTS_DIR / "report_ready_ablation_barplot.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

melted = metrics_df.melt(
    id_vars=["experiment", "question", "model_family"],
    value_vars=["best_val_acc", "final_val_acc"],
    var_name="metric",
    value_name="value",
)
plt.figure(figsize=(14, 6))
sns.barplot(data=melted, x="experiment", y="value", hue="metric")
plt.xticks(rotation=35, ha="right")
plt.ylim(0, 1)
plt.title("Ablation Validation Metrics")
plt.tight_layout()
compat_plot_path = RESULTS_DIR / "ablation_metrics_barplot.png"
plt.savefig(compat_plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {compat_plot_path}")

best = metrics_df.sort_values("best_val_acc", ascending=False).iloc[0]
print(f"Best logged setting: {best['experiment']} ({best['best_val_acc']:.4f})")

<Figure size 1000x500 with 1 Axes>

Saved: c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\ablation_results\report_ready_ablation_barplot.png


<Figure size 1400x600 with 1 Axes>

Saved: c:\Users\zz\Desktop\school\SCM 256\final_proj\Loadsmart_final_report\notebook2&3\ablation_results\ablation_metrics_barplot.png
Best logged setting: 08_yolov8s_640_baseline_negatives (0.9198)


Outputs written to `ablation_results/`:

- `report_ready_ablation_metrics.csv`
- `report_ready_ablation_claims.csv`
- `report_ready_ablation_barplot.png`
- `ablation_metrics.csv`
- `ablation_pairwise_deltas.csv`
- `ablation_metrics_barplot.png`

Training logs are written to `runs/ablation_study/`. The ResNet history is written to `yolo_weights/resnet18_density_v2_history.csv`.